# Resume G — chọn Drive

pred_G có sẵn thì SKIP file cũ, không ghi đè. n<200 thì dừng.

In [ ]:
from google.colab import drive
import os, zipfile, json
from pathlib import Path
import torch

print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU → Save → Restart'

print('\n=== CHON TAI KHOAN DRIVE ===')
print('Popup Google: avatar → Use another account → tk CU → Allow\n')

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

def find_all(name):
    hits = []
    for r in (Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')):
        if r.exists():
            hits.extend(sorted(r.rglob(name)))
    return hits

def pick(title, paths):
    assert paths, f'Khong thay {title} — sai tk. Chay lai, chon tk khac.'
    if len(paths) == 1:
        print(f'{title}: {paths[0]}')
        return paths[0]
    print(f'\nNhieu {title}, nhap so:')
    for i, p in enumerate(paths, 1):
        print(f'  {i}. {p}')
    return paths[int(input('Chon: ').strip()) - 1]

zpath = pick('slatriage_adapters.zip', find_all('slatriage_adapters.zip'))
gpath = pick('pred_G.jsonl', find_all('pred_G.jsonl'))
out = gpath.parent
rows = []
for line in gpath.open(encoding='utf-8'):
    if line.strip():
        rows.append(json.loads(line))
srcs = {r.get('source') for r in rows if r.get('source')}
print(f'pred_G n={len(rows)} sources={len(srcs)} path={gpath}')
assert len(rows) >= 200, f'sai pred_G (n={len(rows)}) — DUNG chay, chon lai Drive cu'
print(f'RESUME: giu {len(rows)} dong, SKIP {len(srcs)} file, chay not {112-len(srcs)} file')
for p in sorted(out.glob('pred_*.jsonl')):
    n = sum(1 for line in p.open(encoding='utf-8') if line.strip())
    print(f'  {p.name}: n={n}')

REPO = 'https://github.com/kham3465/Qu-n-Tr-An-Ninh-M-ng.git'
ROOT = Path('/content/slatriage')
if not (ROOT / '.git').exists():
    !git clone --depth 1 {REPO} {ROOT}
!git -C {ROOT} fetch origin master
!git -C {ROOT} reset --hard origin/master

ad = Path('/content/adapters')
if not (ad / 'R' / 'adapter_config.json').exists():
    ad.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(zpath).extractall(ad)
    if not (ad / 'R' / 'adapter_config.json').exists():
        ad = next(ad.rglob('R/adapter_config.json')).parent.parent

os.environ['SLATRIAGE_ADAPTERS'] = str(ad)
os.environ['SLATRIAGE_PRED_DIR'] = str(out)
os.environ['SLATRIAGE_CONFIGS'] = 'G'
os.chdir(ROOT)
!pip install -q -r requirements-train.txt
!python scripts/colab_eval_rest.py